# Deep Learning Descriptors — Classifier-based (CNN + ViT)

Approche baseline : on entraîne un **ResNet50** (CNN) et un **ViT-Small** (ViT) en mode
classification pure (Cross-Entropy) sur les 96 classes marque+modèle, puis on extrait
l'**avant-dernière couche** (features pré-logits) comme descripteur d'image.

**Différences avec le notebook Metric Learning :**
- Loss classification simple (CrossEntropy), pas d'ArcFace
- Pas de K-Fold : simple train/val split stratifié
- Modèles plus légers (ResNet50, ViT-Small) — entraînement rapide
- Benchmark identique : temps d'indexation, taille descripteur, temps de recherche,
  R/P/AP/mAP par query, 5 distances, PCA 3D


## 1. Imports & config

In [1]:
import os, gc, time, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import timm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
SEED = 123
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
GPU: NVIDIA GeForce RTX 5070


## 2. Dataset (96 classes marque+modèle)

In [2]:
data_dir = Path("../data/raw/Cars")

queries_to_exclude = {
    "0_1_BMW_X3_207.jpg", "0_0_BMW_Serie3Berline_74.jpg", "0_2_BMW_i8_299.jpg",
    "2_0_Volkswagen_Touareg_2822.jpg", "2_4_Volkswagen_Polo_3463.jpg", "2_9_Volkswagen_T-Roc_4209.jpg",
    "4_2_Opel_vivarofourgon_5999.jpg", "4_4_Opel_Insignatourer_6353.jpg", "4_9_Opel_zafiralife_6887.jpg",
    "6_0_Hyundai_Nexo_8282.jpg", "6_3_Hyundai_i10_8837.jpg", "6_5_Hyundai_i30_9125.jpg",
    "8_1_Ford_Puma_11276.jpg", "8_5_Ford_Explorer_11897.jpg", "8_6_Ford_Focus_11951.jpg",
}

all_image_paths = [f for f in data_dir.glob("*.jpg") if f.name not in queries_to_exclude]
all_labels = [f"{p.stem.split('_')[0]}_{p.stem.split('_')[1]}" for p in all_image_paths]

label_encoder = LabelEncoder()
all_labels_encoded = label_encoder.fit_transform(all_labels)
NUM_CLASSES = len(label_encoder.classes_)
print(f"Images: {len(all_image_paths)} | Classes: {NUM_CLASSES}")

X = np.array(all_image_paths)
y = np.array(all_labels_encoded)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
print(f"Train: {len(X_train)} | Val: {len(X_val)}")


Images: 9985 | Classes: 96
Train: 7988 | Val: 1997


## 3. Dataset, transforms & loaders

In [3]:
class CarsDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, int(self.labels[i])


IMG_SIZE = 224
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

train_loader = DataLoader(CarsDataset(X_train, y_train, train_tf),
                          batch_size=32, shuffle=True, num_workers=2,
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(CarsDataset(X_val, y_val, val_tf),
                          batch_size=64, shuffle=False, num_workers=2,
                          pin_memory=True)


## 4. Modèles classifier — ResNet50 & ViT-Small

Les deux modèles exposent une méthode `extract_features(x)` qui retourne l'**avant-dernière couche**
(pré-logits) pour l'utiliser comme descripteur après entraînement.

In [4]:
class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.feat_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Linear(self.feat_dim, num_classes)
    def forward(self, x):
        f = self.backbone(x)
        return self.classifier(f)
    @torch.no_grad()
    def extract_features(self, x):
        return self.backbone(x)


class ViTSmallClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = timm.create_model(
            "vit_small_patch16_224", pretrained=True, num_classes=0, global_pool="token")
        self.feat_dim = self.backbone.num_features   # 384
        self.classifier = nn.Linear(self.feat_dim, num_classes)
    def forward(self, x):
        f = self.backbone(x)
        return self.classifier(f)
    @torch.no_grad()
    def extract_features(self, x):
        return self.backbone(x)


## 5. Boucle d'entraînement générique

In [5]:
def train_classifier(model, num_epochs=25, lr=3e-4, save_path="models/clf.pth"):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    best_acc = 0.0
    for epoch in range(num_epochs):
        model.train(); t0 = time.time(); run_loss = 0.0
        for imgs, labels in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=scaler is not None):
                logits = model(imgs); loss = criterion(logits, labels)
            if scaler is not None:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                loss.backward(); optimizer.step()
            run_loss += loss.item()

        model.eval(); correct = total = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device); labels = labels.to(device)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=scaler is not None):
                    preds = model(imgs).argmax(1)
                correct += (preds == labels).sum().item(); total += labels.size(0)
        val_acc = correct / total
        scheduler.step()
        print(f"Epoch {epoch+1:2d}/{num_epochs} | loss {run_loss/len(train_loader):.4f} | val_acc {val_acc*100:.2f}% | {time.time()-t0:.1f}s")
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({"model_state": model.state_dict(), "val_acc": val_acc, "epoch": epoch+1}, save_path)
    print(f"Best val_acc: {best_acc*100:.2f}%")
    return save_path


## 6. Entraînement ResNet50

In [6]:
resnet_model = ResNet50Classifier(NUM_CLASSES)
resnet_path = train_classifier(resnet_model, num_epochs=25, lr=3e-4,
                               save_path="models/resnet50_clf.pth")
del resnet_model; gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()


Epoch  1/25 | loss 3.4016 | val_acc 49.52% | 18.9s
Epoch  2/25 | loss 1.6984 | val_acc 72.61% | 17.4s
Epoch  3/25 | loss 1.3065 | val_acc 73.06% | 16.4s
Epoch  4/25 | loss 1.1177 | val_acc 78.42% | 16.7s
Epoch  5/25 | loss 1.0242 | val_acc 80.62% | 17.4s
Epoch  6/25 | loss 0.9671 | val_acc 78.17% | 17.8s
Epoch  7/25 | loss 0.9454 | val_acc 83.98% | 17.4s
Epoch  8/25 | loss 0.9114 | val_acc 82.57% | 17.3s
Epoch  9/25 | loss 0.8930 | val_acc 83.93% | 17.3s
Epoch 10/25 | loss 0.8683 | val_acc 86.03% | 17.0s
Epoch 11/25 | loss 0.8549 | val_acc 86.53% | 16.9s
Epoch 12/25 | loss 0.8406 | val_acc 87.08% | 17.5s
Epoch 13/25 | loss 0.8302 | val_acc 87.08% | 16.8s
Epoch 14/25 | loss 0.8207 | val_acc 87.48% | 16.9s
Epoch 15/25 | loss 0.8198 | val_acc 87.23% | 16.7s
Epoch 16/25 | loss 0.8131 | val_acc 87.68% | 16.7s
Epoch 17/25 | loss 0.8079 | val_acc 88.68% | 16.7s
Epoch 18/25 | loss 0.8034 | val_acc 88.33% | 16.8s
Epoch 19/25 | loss 0.7992 | val_acc 88.63% | 16.9s
Epoch 20/25 | loss 0.7962 | val

## 7. Entraînement ViT-Small

In [7]:
vit_model = ViTSmallClassifier(NUM_CLASSES)
vit_path = train_classifier(vit_model, num_epochs=25, lr=1e-4,
                            save_path="models/vit_small_clf.pth")
del vit_model; gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()


Epoch  1/25 | loss 4.4167 | val_acc 18.78% | 17.2s
Epoch  2/25 | loss 2.6589 | val_acc 55.63% | 17.2s
Epoch  3/25 | loss 1.6733 | val_acc 68.70% | 17.0s
Epoch  4/25 | loss 1.3071 | val_acc 72.96% | 17.0s
Epoch  5/25 | loss 1.1392 | val_acc 75.21% | 16.8s
Epoch  6/25 | loss 1.0434 | val_acc 78.12% | 16.9s
Epoch  7/25 | loss 0.9866 | val_acc 79.47% | 16.9s
Epoch  8/25 | loss 0.9461 | val_acc 80.47% | 16.8s
Epoch  9/25 | loss 0.9210 | val_acc 81.32% | 16.8s
Epoch 10/25 | loss 0.9026 | val_acc 83.07% | 17.1s
Epoch 11/25 | loss 0.8899 | val_acc 83.22% | 16.8s
Epoch 12/25 | loss 0.8769 | val_acc 81.97% | 16.8s
Epoch 13/25 | loss 0.8608 | val_acc 84.63% | 16.8s
Epoch 14/25 | loss 0.8559 | val_acc 84.73% | 17.1s
Epoch 15/25 | loss 0.8455 | val_acc 84.63% | 17.0s
Epoch 16/25 | loss 0.8430 | val_acc 84.93% | 17.1s
Epoch 17/25 | loss 0.8358 | val_acc 84.68% | 16.9s
Epoch 18/25 | loss 0.8306 | val_acc 86.18% | 17.0s
Epoch 19/25 | loss 0.8276 | val_acc 84.98% | 16.8s
Epoch 20/25 | loss 0.8249 | val

## 8. Extraction des descripteurs (avant-dernière couche) + L2-norm

In [8]:
def load_model(factory, ckpt_path):
    m = factory()
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    m.load_state_dict(ckpt["model_state"])
    return m.to(device).eval()


@torch.no_grad()
def extract_descriptors(model, dataloader):
    model.eval(); feats, labs = [], []
    for imgs, labels in dataloader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device.type=="cuda"):
            f = model.extract_features(imgs)
        f = F.normalize(f.float(), p=2, dim=1)
        feats.append(f.cpu().numpy())
        labs.append(labels.numpy() if torch.is_tensor(labels) else np.asarray(labels))
    return np.concatenate(feats), np.concatenate(labs)


query_filenames = [
    "0_1_BMW_X3_207.jpg","0_0_BMW_Serie3Berline_74.jpg","0_2_BMW_i8_299.jpg",
    "2_0_Volkswagen_Touareg_2822.jpg","2_4_Volkswagen_Polo_3463.jpg","2_9_Volkswagen_T-Roc_4209.jpg",
    "4_2_Opel_vivarofourgon_5999.jpg","4_4_Opel_Insignatourer_6353.jpg","4_9_Opel_zafiralife_6887.jpg",
    "6_0_Hyundai_Nexo_8282.jpg","6_3_Hyundai_i10_8837.jpg","6_5_Hyundai_i30_9125.jpg",
    "8_1_Ford_Puma_11276.jpg","8_5_Ford_Explorer_11897.jpg","8_6_Ford_Focus_11951.jpg",
]
query_paths = [data_dir / q for q in query_filenames]
query_labels = np.array([label_encoder.transform([f"{q.split('_')[0]}_{q.split('_')[1]}"])[0]
                         for q in query_filenames])

gallery_loader = DataLoader(CarsDataset(X, y, val_tf), batch_size=64, shuffle=False,
                            num_workers=2, pin_memory=True)
query_loader   = DataLoader(CarsDataset(np.array(query_paths), query_labels, val_tf),
                            batch_size=32, shuffle=False, num_workers=2, pin_memory=True)


resnet = load_model(lambda: ResNet50Classifier(NUM_CLASSES), resnet_path)
t0 = time.time()
resnet_gallery, gallery_labels = extract_descriptors(resnet, gallery_loader)
resnet_index_time = time.time() - t0
resnet_query, q_labels = extract_descriptors(resnet, query_loader)
print(f"ResNet50 gallery: {resnet_gallery.shape} in {resnet_index_time:.1f}s  | query: {resnet_query.shape}")
del resnet; gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()


vit = load_model(lambda: ViTSmallClassifier(NUM_CLASSES), vit_path)
t0 = time.time()
vit_gallery, _ = extract_descriptors(vit, gallery_loader)
vit_index_time = time.time() - t0
vit_query, _   = extract_descriptors(vit, query_loader)
print(f"ViT-Small gallery: {vit_gallery.shape} in {vit_index_time:.1f}s | query: {vit_query.shape}")
del vit; gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()


ResNet50 gallery: (9985, 2048) in 11.6s  | query: (15, 2048)


ViT-Small gallery: (9985, 384) in 11.4s | query: (15, 384)


## 9. Métriques & distances

In [9]:
def average_precision(retrieved, true_label, k=None):
    if k is not None: retrieved = retrieved[:k]
    hits = (retrieved == true_label).astype(np.float32)
    if hits.sum() == 0: return 0.0
    precisions = np.cumsum(hits) / (np.arange(len(hits)) + 1)
    return float((precisions * hits).sum() / hits.sum())


def dist_euclidean(q, g):   return cdist(q, g, metric="euclidean")
def dist_cosine(q, g):      return 1.0 - q @ g.T   # L2-normalisés
def dist_correlation(q, g): return cdist(q, g, metric="correlation")
def dist_chi_square(q, g):
    shift = min(q.min(), g.min()); q2 = q - shift + 1e-8; g2 = g - shift + 1e-8
    out = np.zeros((q2.shape[0], g2.shape[0]), dtype=np.float32)
    for i in range(q2.shape[0]):
        num = (q2[i:i+1] - g2) ** 2; den = q2[i:i+1] + g2 + 1e-8
        out[i] = 0.5 * (num / den).sum(axis=1)
    return out
def dist_bhattacharyya(q, g):
    shift = min(q.min(), g.min()); q2 = q - shift + 1e-8; g2 = g - shift + 1e-8
    q2 /= q2.sum(axis=1, keepdims=True); g2 /= g2.sum(axis=1, keepdims=True)
    bc = np.clip(np.sqrt(q2) @ np.sqrt(g2).T, 1e-8, 1.0)
    return -np.log(bc)

DISTANCES = {
    "Euclidean": dist_euclidean, "Cosine": dist_cosine,
    "Correlation": dist_correlation, "Chi-square": dist_chi_square,
    "Bhattacharyya": dist_bhattacharyya,
}


def avg_search_time(qf, gf, dist_fn, repeats=3):
    nq = qf.shape[0]; times = []
    for _ in range(repeats):
        t0 = time.time()
        for i in range(nq): _ = dist_fn(qf[i:i+1], gf)
        times.append((time.time()-t0)/nq)
    return float(np.mean(times))


## 10. Table 3 — Performance des descripteurs

In [10]:
descriptors = {
    "ResNet50 (pre-logits)":   (resnet_query, resnet_gallery, resnet_index_time),
    "ViT-Small (pre-logits)":  (vit_query,    vit_gallery,    vit_index_time),
}

rows = []
for name, (qf, gf, idx_t) in descriptors.items():
    size_mb = gf.nbytes / (1024**2)
    for dname, dfn in DISTANCES.items():
        ts = avg_search_time(qf, gf, dfn, repeats=3)
        rows.append({
            "Descriptor": name, "Distance": dname,
            "Indexing Time (s)": f"{idx_t:.1f}",
            "Descriptor Size (MB)": f"{size_mb:.2f}",
            "Dim": gf.shape[1],
            "Avg Search Time/image (ms)": f"{ts*1000:.2f}",
        })

table3 = pd.DataFrame(rows)
print("=== Table 3 — Descriptor Performance ===")
print(table3.to_string(index=False))


=== Table 3 — Descriptor Performance ===
            Descriptor      Distance Indexing Time (s) Descriptor Size (MB)  Dim Avg Search Time/image (ms)
 ResNet50 (pre-logits)     Euclidean              11.6                78.01 2048                      15.51
 ResNet50 (pre-logits)        Cosine              11.6                78.01 2048                       1.91
 ResNet50 (pre-logits)   Correlation              11.6                78.01 2048                      39.64
 ResNet50 (pre-logits)    Chi-square              11.6                78.01 2048                      43.83
 ResNet50 (pre-logits) Bhattacharyya              11.6                78.01 2048                      35.19
ViT-Small (pre-logits)     Euclidean              11.4                14.63  384                       2.37
ViT-Small (pre-logits)        Cosine              11.4                14.63  384                       0.05
ViT-Small (pre-logits)   Correlation              11.4                14.63  384               

## 11. Table 4 — Métriques par query (Top-50 / Top-100)

In [11]:
def build_per_query_table(query_feats, gallery_feats, q_labels, g_labels,
                          dist_fn=dist_cosine, top_ks=(50, 100)):
    dist = dist_fn(query_feats, gallery_feats)
    order = np.argsort(dist, axis=1)
    rows = []; ap_running = {k: [] for k in top_ks}
    for i in range(len(q_labels)):
        ranked = g_labels[order[i]]
        n_rel = int((g_labels == q_labels[i]).sum())
        row = {"Query": f"R{i+1}"}
        for k in top_ks:
            hits = (ranked[:k] == q_labels[i]).astype(np.float32)
            tp = int(hits.sum())
            R = tp / max(1, n_rel); P = tp / k
            AP = average_precision(ranked, q_labels[i], k=k)
            ap_running[k].append(AP)
            row[f"R@{k}"]   = f"{R*100:.2f}%"
            row[f"P@{k}"]   = f"{P*100:.2f}%"
            row[f"AP@{k}"]  = f"{AP*100:.2f}%"
            row[f"mAP@{k}"] = f"{np.mean(ap_running[k])*100:.2f}%"
        rows.append(row)
    final = {"Query": "MEAN"}
    for k in top_ks:
        for met in ("R","P","AP"):
            final[f"{met}@{k}"] = f"{np.mean([float(r[f'{met}@{k}'][:-1]) for r in rows]):.2f}%"
        final[f"mAP@{k}"] = f"{np.mean(ap_running[k])*100:.2f}%"
    rows.append(final)
    return pd.DataFrame(rows), order


print("=== ResNet50 (Cosine) ===")
tbl_resnet, order_resnet = build_per_query_table(resnet_query, resnet_gallery, q_labels, gallery_labels)
print(tbl_resnet.to_string(index=False))

print("\n=== ViT-Small (Cosine) ===")
tbl_vit, order_vit = build_per_query_table(vit_query, vit_gallery, q_labels, gallery_labels)
print(tbl_vit.to_string(index=False))

os.makedirs("reports", exist_ok=True)
table3.to_csv("reports/dl_table3_descriptor_performance.csv", index=False)
tbl_resnet.to_csv("reports/dl_table4_resnet50.csv", index=False)
tbl_vit.to_csv("reports/dl_table4_vit_small.csv", index=False)
print("\nSaved to reports/")


=== ResNet50 (Cosine) ===
Query   R@50    P@50   AP@50  mAP@50   R@100   P@100  AP@100 mAP@100
   R1 31.06% 100.00% 100.00% 100.00%  62.11% 100.00% 100.00% 100.00%
   R2 50.51% 100.00% 100.00% 100.00%  89.90%  89.00%  99.28%  99.64%
   R3 36.23% 100.00% 100.00% 100.00%  72.46% 100.00% 100.00%  99.76%
   R4 50.51% 100.00% 100.00% 100.00%  97.98%  97.00% 100.00%  99.82%
   R5 50.51% 100.00% 100.00% 100.00%  95.96%  95.00%  99.94%  99.84%
   R6 50.51% 100.00% 100.00% 100.00% 100.00%  99.00%  99.99%  99.87%
   R7 50.51% 100.00% 100.00% 100.00%  96.97%  96.00%  99.88%  99.87%
   R8 49.49%  98.00%  98.16%  99.77%  86.87%  86.00%  97.21%  99.54%
   R9 44.44%  88.00%  88.33%  98.50%  83.84%  83.00%  86.46%  98.08%
  R10 50.51% 100.00% 100.00%  98.65%  98.99%  98.00% 100.00%  98.28%
  R11 34.01% 100.00% 100.00%  98.77%  68.03% 100.00% 100.00%  98.43%
  R12 50.51% 100.00% 100.00%  98.87%  89.90%  89.00%  99.72%  98.54%
  R13 35.97% 100.00% 100.00%  98.96%  71.94% 100.00% 100.00%  98.65%
  R14 47

## 12. Visualisation 3D de l'espace latent

In [12]:
import umap

def plot_umap_3d(gallery_feats, query_feats, title,
                 n_neighbors=30, min_dist=0.1):
    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric='cosine',
        random_state=SEED,
        verbose=False,
    )
    g3 = reducer.fit_transform(gallery_feats)
    q3 = reducer.transform(query_feats)
    print(f"{title}: UMAP 3D (n_neighbors={n_neighbors}, min_dist={min_dist})")

    df_g = pd.DataFrame({
        "UMAP1": g3[:,0], "UMAP2": g3[:,1], "UMAP3": g3[:,2],
        "Class": label_encoder.inverse_transform(gallery_labels),
    })
    fig = px.scatter_3d(df_g, x="UMAP1", y="UMAP2", z="UMAP3", color="Class",
                        opacity=0.55,
                        title=f"{title} — UMAP 3D (cosine)")
    fig.update_traces(marker=dict(size=3))
    fig.add_trace(go.Scatter3d(
        x=q3[:,0], y=q3[:,1], z=q3[:,2],
        mode="markers+text",
        marker=dict(size=8, color="black", symbol="diamond",
                    line=dict(width=2, color="white")),
        text=[f"R{i+1}" for i in range(len(q3))],
        textposition="top center", name="Queries",
    ))
    fig.update_layout(
        scene=dict(
            xaxis_title="UMAP 1",
            yaxis_title="UMAP 2",
            zaxis_title="UMAP 3"),
        width=1000, height=700, legend=dict(font=dict(size=8)))
    fig.show()


plot_umap_3d(resnet_gallery, resnet_query, "ResNet50 classifier (pre-logits)")
plot_umap_3d(vit_gallery,    vit_query,    "ViT-Small classifier (pre-logits)")

ResNet50 classifier (pre-logits): UMAP 3D (n_neighbors=30, min_dist=0.1)


ViT-Small classifier (pre-logits): UMAP 3D (n_neighbors=30, min_dist=0.1)
